# Project 63 — Local RAG A/B Testing Notebook

**Compare chunking strategies and prompt variants for RAG pipelines — locally.**

| Component | Choice |
|-----------|--------|
| LLM runtime | Ollama (qwen3:8b) |
| Framework | LangChain |
| Vector store | ChromaDB |
| Interface | Jupyter notebook |

## 1 · What You Will Learn

1. How to set up **controlled A/B experiments** for RAG
2. How to compare **chunking strategies** (small vs large)
3. How to compare **retrieval parameters** (top-k)
4. How to compare **generation prompts** for the same context
5. How to **measure RAG quality** with automated metrics

## 2 · Why This Project Matters

RAG performance depends on many interacting factors. Without systematic A/B
testing, you cannot know which combination works best.

## 3 · Setup

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb

## 4 · Imports

In [ ]:
import json, time
import pandas as pd
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

MODEL = 'qwen3:8b'
llm = ChatOllama(model=MODEL, temperature=0.1)
embeddings = OllamaEmbeddings(model=MODEL)
print(f'LLM and embeddings ready: {MODEL}')

## 5 · Knowledge Base

In [ ]:
DOCUMENTS = [
    'Python is a high-level programming language created by Guido van Rossum in 1991. It supports procedural, OO, and functional paradigms.',
    'Machine learning enables systems to learn from data. Common algorithms include decision trees, random forests, and neural networks.',
    'Docker containers share the host OS kernel, making them lighter than VMs. Core concepts: images, containers, registries.',
    'PostgreSQL supports ACID transactions, complex queries, JSON types, and full-text search.',
    'Git is a distributed VCS created by Linus Torvalds in 2005. Key concepts: commits, branches, merges, remotes.',
    'REST APIs use HTTP methods (GET, POST, PUT, DELETE) and typically return JSON.',
]
print(f'Knowledge base: {len(DOCUMENTS)} documents')

## 6 · A/B Configurations

In [ ]:
CONFIG_A = {'name': 'A_small_chunks', 'chunk_size': 100, 'chunk_overlap': 20, 'top_k': 3,
            'prompt': ChatPromptTemplate.from_messages([
                ('system', 'Answer based on context. If unsure, say so.'),
                ('human', 'Context: {context}\nQuestion: {question}')])}
CONFIG_B = {'name': 'B_large_chunks', 'chunk_size': 300, 'chunk_overlap': 50, 'top_k': 2,
            'prompt': ChatPromptTemplate.from_messages([
                ('system', 'Answer using ONLY the context. Cite evidence.'),
                ('human', 'Context: {context}\nQuestion: {question}')])}
print(f'Config A: chunk={CONFIG_A["chunk_size"]}, k={CONFIG_A["top_k"]}')
print(f'Config B: chunk={CONFIG_B["chunk_size"]}, k={CONFIG_B["top_k"]}')

## 7 · Build RAG Pipelines

In [ ]:
def build_pipeline(config, documents):
    splitter = RecursiveCharacterTextSplitter(chunk_size=config['chunk_size'], chunk_overlap=config['chunk_overlap'])
    chunks = splitter.split_documents([Document(page_content=d) for d in documents])
    vs = Chroma.from_documents(chunks, embeddings, collection_name=config['name'])
    return vs.as_retriever(search_kwargs={'k': config['top_k']}), config['prompt'], len(chunks)

ret_a, prompt_a, n_a = build_pipeline(CONFIG_A, DOCUMENTS)
ret_b, prompt_b, n_b = build_pipeline(CONFIG_B, DOCUMENTS)
print(f'A: {n_a} chunks, B: {n_b} chunks')

## 8 · Run A/B Test

In [ ]:
QUESTIONS = ['What paradigms does Python support?', 'How do Docker containers differ from VMs?',
             'What are Git core concepts?', 'What HTTP methods do REST APIs use?']

def run_rag(q, retriever, prompt_tmpl):
    t0 = time.time()
    docs = retriever.invoke(q)
    context = '\n'.join([d.page_content for d in docs])
    t1 = time.time()
    answer = (prompt_tmpl | llm | StrOutputParser()).invoke({'context': context, 'question': q})
    return {'answer': answer, 'latency': round(time.time()-t0, 3), 'words': len(answer.split())}

ab_results = []
for q in QUESTIONS:
    print(f'Testing: {q[:50]}...')
    ab_results.append({'question': q, 'A': run_rag(q, ret_a, prompt_a), 'B': run_rag(q, ret_b, prompt_b)})
print(f'Completed {len(ab_results)} comparisons')

## 9 · Judge and Compare

In [ ]:
judge_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Compare two answers. Return JSON: {"winner": "A"/"B"/"tie", "A_score": 1-5, "B_score": 1-5}'),
    ('human', 'Q: {question}\nA: {a}\nB: {b}'),
])
judge_chain = judge_prompt | ChatOllama(model=MODEL, temperature=0.1) | StrOutputParser()

for item in ab_results:
    raw = judge_chain.invoke({'question': item['question'], 'a': item['A']['answer'], 'b': item['B']['answer']})
    try:
        s, e = raw.find('{'), raw.rfind('}') + 1
        item['judgment'] = json.loads(raw[s:e]) if s >= 0 else {'winner': 'tie'}
    except: item['judgment'] = {'winner': 'tie'}
    print(f'{item["question"][:40]:40s} Winner: {item["judgment"].get("winner","?")}')

## 10 · Results Summary

In [ ]:
rows = [{'question': i['question'][:35], 'A_time': i['A']['latency'], 'B_time': i['B']['latency'],
         'A_words': i['A']['words'], 'B_words': i['B']['words'],
         'winner': i.get('judgment',{}).get('winner','?')} for i in ab_results]
df = pd.DataFrame(rows)
print(df.to_string(index=False))

## 11 · Save Results

In [ ]:
with open('rag_ab_results.json', 'w') as f:
    json.dump(ab_results, f, indent=2, default=str)
print('Saved.')

## 12 · Limitations & Improvements

- Small corpus (6 docs) does not stress-test retrieval
- Should compare embedding models too
- Single run — repeat for variance estimation
- Add hybrid BM25 + dense retrieval as Config C

## 13 · Key Takeaways

- Chunking strategy significantly impacts retrieval quality
- A/B testing gives data-driven RAG design decisions
- LLM-as-judge automates answer comparison
- ChromaDB + Ollama = fully local RAG experimentation